In [1]:
import sys
sys.executable

'H:\\Documents\\2. Perso\\github\\mlflow\\.venv\\Scripts\\python.exe'

In [2]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
csv_path = PROJECT_ROOT / "data" / "raw" / "adult-census.csv"

In [3]:
import pandas as pd

adult_census = pd.read_csv(csv_path)
adult_census = adult_census.drop(columns="education.num")

target_name = "income"
target = adult_census[target_name]
data = adult_census.drop(columns=[target_name])

In [4]:
from sklearn.compose import make_column_selector as selector

numerical_columns_selector = selector(dtype_exclude=object)
categorical_columns_selector = selector(dtype_include=object)

numerical_columns = numerical_columns_selector(data)
categorical_columns = categorical_columns_selector(data)

# Modèle #1 — LogisticRegression

In [5]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import make_column_transformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

categorical_preprocessor = OneHotEncoder(handle_unknown="ignore")
numerical_preprocessor = StandardScaler()

preprocessor = make_column_transformer(
    (categorical_preprocessor, categorical_columns),
    (numerical_preprocessor, numerical_columns),
)

model_1 = make_pipeline(preprocessor, LogisticRegression(max_iter=500))

In [6]:
from sklearn.model_selection import train_test_split

data_train, data_test, target_train, target_test = train_test_split(
    data, target, random_state=42
)

In [7]:
%%time
_ = model_1.fit(data_train, target_train)

model_1.score(data_test, target_test)

CPU times: total: 297 ms
Wall time: 319 ms


0.8482987347991647

In [8]:
from sklearn.model_selection import cross_validate
import mlflow

# Set experiment (do this once per notebook)
mlflow.set_experiment("adult_census_day1_tracking")

with mlflow.start_run(run_name="model_1_logreg_ohe_scaler_cv"):
    cv_results_1 = cross_validate(model_1, data, target, cv=5)
    scores_1 = cv_results_1["test_score"]

    mean_acc_1 = float(scores_1.mean())
    std_acc_1 = float(scores_1.std())

    mlflow.log_param("model_id", "model_1")
    mlflow.log_param("model_family", "LogisticRegression")
    mlflow.log_param("categorical_encoder", "OneHotEncoder(ignore)")
    mlflow.log_param("numerical_scaler", "StandardScaler")
    mlflow.log_param("cv_folds", 5)

    mlflow.log_metric("cv_accuracy_mean", mean_acc_1)
    mlflow.log_metric("cv_accuracy_std", std_acc_1)

    print(
        "The mean cross-validation accuracy is: "
        f"{mean_acc_1:.3f} ± {std_acc_1:.3f}"
    )

2026/01/13 16:21:53 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/01/13 16:21:53 INFO mlflow.store.db.utils: Updating database tables
2026/01/13 16:21:53 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/13 16:21:53 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/01/13 16:21:53 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/13 16:21:53 INFO alembic.runtime.migration: Will assume non-transactional DDL.


The mean cross-validation accuracy is: 0.826 ± 0.031


# Modèle #2 — HistGradientBoosting

In [9]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

categorical_preprocessor = OrdinalEncoder(
    handle_unknown="use_encoded_value", unknown_value=-1
)

preprocessor = make_column_transformer(
    (categorical_preprocessor, categorical_columns),
    remainder="passthrough",
)

model_2 = make_pipeline(preprocessor, HistGradientBoostingClassifier())

In [10]:
%%time
_ = model_2.fit(data_train, target_train)

model_2.score(data_test, target_test)

CPU times: total: 1.77 s
Wall time: 8.29 s


0.8727429062768701

In [11]:
from sklearn.model_selection import cross_validate

with mlflow.start_run(run_name="model_2_hgb_ordinal_cv"):
    cv_results = cross_validate(model_2, data, target, cv=5)
    scores = cv_results["test_score"]

    mean_acc_2 = float(scores.mean())
    std_acc_2 = float(scores.std())

    mlflow.log_param("model_id", "model_2")
    mlflow.log_param("model_family", "HistGradientBoostingClassifier")
    mlflow.log_param("categorical_encoder", "OrdinalEncoder(unknown=-1)")
    mlflow.log_param("cv_folds", 5)

    mlflow.log_metric("cv_accuracy_mean", mean_acc_2)
    mlflow.log_metric("cv_accuracy_std", std_acc_2)

    print(
        "The mean cross-validation accuracy is: "
        f"{mean_acc_2:.3f} ± {std_acc_2:.3f}"
    )

The mean cross-validation accuracy is: 0.811 ± 0.030


# Modèle #3 — RandomForest

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline

model_3 = make_pipeline(
    preprocessor,
    RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
    ),
)

In [14]:
%%time
_ = model_3.fit(data_train, target_train)

model_3.score(data_test, target_test)

CPU times: total: 13.6 s
Wall time: 4.07 s


0.8551774966220366

In [15]:
from sklearn.model_selection import cross_validate

with mlflow.start_run(run_name="model_3_rf_ohe_cv"):
    cv_results_3 = cross_validate(model_3, data, target, cv=5)
    scores_3 = cv_results_3["test_score"]

    mean_acc_3 = float(scores_3.mean())
    std_acc_3 = float(scores_3.std())

    mlflow.log_param("model_id", "model_3")
    mlflow.log_param("model_family", "RandomForestClassifier")
    mlflow.log_param("categorical_encoder", "OneHotEncoder(ignore)")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("cv_folds", 5)

    mlflow.log_metric("cv_accuracy_mean", mean_acc_3)
    mlflow.log_metric("cv_accuracy_std", std_acc_3)

    print(
        "The mean cross-validation accuracy is: "
        f"{mean_acc_3:.3f} ± {std_acc_3:.3f}"
    )

The mean cross-validation accuracy is: 0.802 ± 0.028


# Modèle #4 — XGBoost

In [16]:
from xgboost import XGBClassifier
from sklearn.pipeline import make_pipeline

model_4 = make_pipeline(
    preprocessor,
    XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    )
)

In [18]:
from sklearn.preprocessing import LabelEncoder

# Encode target for XGBoost (expects numeric classes like 0/1)
le = LabelEncoder()
target_enc = le.fit_transform(target)
target_train_enc = le.fit_transform(target_train)
target_test_enc = le.transform(target_test)

In [19]:
%%time
_ = model_4.fit(data_train, target_train_enc)

model_4.score(data_test, target_test_enc)

CPU times: total: 3.41 s
Wall time: 909 ms


0.8683208451050239

In [20]:
from sklearn.model_selection import cross_validate

with mlflow.start_run(run_name="model_4_xgb_ohe_cv"):
    cv_results_4 = cross_validate(model_4, data, target_enc, cv=5)
    scores_4 = cv_results_4["test_score"]

    mean_acc_4 = float(scores_4.mean())
    std_acc_4 = float(scores_4.std())

    mlflow.log_param("model_id", "model_4")
    mlflow.log_param("model_family", "XGBClassifier")
    mlflow.log_param("categorical_encoder", "OneHotEncoder(ignore)")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)
    mlflow.log_param("eval_metric", "logloss")
    mlflow.log_param("random_state", 42)
    mlflow.log_param("cv_folds", 5)

    # Keep track of how labels were encoded (0/1)
    mlflow.log_param("target_mapping", {cls: int(i) for i, cls in enumerate(le.classes_)})

    mlflow.log_metric("cv_accuracy_mean", mean_acc_4)
    mlflow.log_metric("cv_accuracy_std", std_acc_4)

    print(
        "The mean cross-validation accuracy is: "
        f"{mean_acc_4:.3f} ± {std_acc_4:.3f}"
    )

The mean cross-validation accuracy is: 0.816 ± 0.026


# Collect CV means + select the best model key

In [21]:
cv_means = {
    "model_1": mean_acc_1,
    "model_2": mean_acc_2,
    "model_3": mean_acc_3,
    "model_4": mean_acc_4,
}

best_key = max(cv_means, key=cv_means.get)
best_key

'model_1'

# Map keys to model objects + pick the champion model

In [22]:
models = {
    "model_1": model_1,
    "model_2": model_2,
    "model_3": model_3,
    "model_4": model_4,
}

best_model = models[best_key]

# Refit champion + log model artifact in MLflow

In [24]:
import mlflow.sklearn

with mlflow.start_run(run_name=f"{best_key}_champion"):
    best_model.fit(data_train, target_train)

    test_acc = best_model.score(data_test, target_test)
    mlflow.log_metric("test_accuracy", float(test_acc))

    mlflow.log_param("selected_model", best_key)
    mlflow.log_param("selection_metric", "cv_accuracy_mean")

    mlflow.sklearn.log_model(best_model, name="model")